# CIDM Lab 2 — Final (Complete)


**Course:** P170M109 Computational Intelligence and Decision Making  
**Author:** _Your Name_  
**Date:** _YYYY-MM-DD_


- **Part 1 – Regression using ANN** (reuses Lab 1 dataset)
- **Part 2 – Image Classification** (from scratch + transfer learning)
- **Part 3 – Object Detection & Segmentation** (YOLOv8; segmentation optional)

> Tip: run Part 1 first; Parts 2 & 3 require your custom images.


In [ ]:
# Environment info
import sys, platform, numpy as np, pandas as pd, matplotlib, sklearn
print('Python:', sys.version.split()[0])
print('OS:', platform.system(), platform.release())
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('sklearn:', sklearn.__version__)
print('matplotlib:', matplotlib.__version__)


## Part 1 — Load Lab 1 Dataset

In [ ]:
from pathlib import Path
import pandas as pd

def find_file_upwards(filename: str):
    p = Path.cwd()
    seen = set()
    while True:
        if (p / filename).exists():
            return (p / filename).resolve()
        seen.add(p)
        if p.parent == p or p.parent in seen:
            break
        p = p.parent
    if (Path('/mnt/data') / filename).exists():
        return (Path('/mnt/data') / filename).resolve()
    return None

candidates = [
    "apartments_for_rent_classified_100K.csv",
    "apartments_for_rent_classified_10K.csv",
]

file_path = None
for name in candidates:
    fp = find_file_upwards(name)
    if fp:
        file_path = fp
        break

assert file_path is not None, "CSV not found. Place the dataset in this folder, a parent folder, or /mnt/data/"
print('Using file:', file_path)

df_raw = pd.read_csv(file_path)
print('Raw shape:', df_raw.shape)
df_raw.head(3)


### Preprocessing (as in Lab 1)

In [ ]:
import numpy as np

df = df_raw.copy()

def to_monthly(price, price_type: str):
    if pd.isna(price): return np.nan
    pt = (price_type or '').strip().lower()
    if pt in ('week','weekly'): return price * 52.0/12.0
    if pt in ('day','daily'): return price * 30.0
    if pt == 'fortnightly': return price * 26.0/12.0
    if pt in ('year','yearly','per year'): return price / 12.0
    if pt in ('hour','hourly'): return price * 24.0 * 30.0
    return price

df['price_monthly'] = [to_monthly(p, t) for p, t in zip(df.get('price'), df.get('price_type'))]

# Basic sanity
if 'square_feet' in df.columns:
    df = df[(df['square_feet'].fillna(0) >= 120) & (df['square_feet'].fillna(0) <= 8000)]
df = df[df['price_monthly'].notna() & (df['price_monthly'] > 0)]

# Clip top 0.5% target
upper = df['price_monthly'].quantile(0.995)
df['price_monthly'] = df['price_monthly'].clip(upper=upper)

# Deduplicate if common columns exist
subset_cols = [c for c in ['title','description','city','region','state','price','square_feet','beds','baths'] if c in df.columns]
if subset_cols:
    df = df.drop_duplicates(subset=subset_cols)

print('Cleaned shape:', df.shape)
df[['price','price_type','price_monthly']].head(5)


### Features & Target

In [ ]:
num_cols = [c for c in ['square_feet','beds','baths'] if c in df.columns]
cat_cols = [c for c in ['state','city','region'] if c in df.columns]
print('Numeric:', num_cols)
print('Categorical:', cat_cols)

features = num_cols + cat_cols
df_model = df[features + ['price_monthly']].dropna(subset=['price_monthly']).copy()
X = df_model[features]
y = df_model['price_monthly'].astype(float)
print('Model df:', df_model.shape)


### Split Train/Val/Test

In [ ]:
from sklearn.model_selection import train_test_split
RANDOM_STATE = 42
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE)
len(X_train), len(X_val), len(X_test)


### Pipelines & Metrics

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

def mape_masked(y_true, y_pred, eps=1e-8):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = np.abs(y_true) > eps
    return (np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]).mean()) * 100.0

def rmse(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return np.sqrt(((y_true - y_pred) ** 2).mean())

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                             ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocess = ColumnTransformer([('num', numeric_pipe, num_cols), ('cat', categorical_pipe, cat_cols)], remainder='drop')


## Baselines — KNN / DecisionTree / RandomForest (same split)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

results_baseline = {}

def fit_eval(name, model):
    pipe = Pipeline([('prep', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    res = {'MAE': mean_absolute_error(y_test, pred),
           'MAPE': mape_masked(y_test, pred),
           'RMSE': rmse(y_test, pred),
           'R2': r2_score(y_test, pred)}
    results_baseline[name] = res
    return res

fit_eval('KNN(k=5)', KNeighborsRegressor(n_neighbors=5))
fit_eval('DecisionTree', DecisionTreeRegressor(random_state=RANDOM_STATE))
fit_eval('RandomForest', RandomForestRegressor(n_estimators=200, min_samples_leaf=1, random_state=RANDOM_STATE))

pd.DataFrame(results_baseline).T


## ANN (MLPRegressor) — Grid Search + Learning Curve

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV

ann = Pipeline([
    ('prep', preprocess),
    ('model', MLPRegressor(
        random_state=RANDOM_STATE,
        early_stopping=True,
        max_iter=1000,
        n_iter_no_change=30,
        validation_fraction=0.15
    ))
])

param_grid = {
    'model__hidden_layer_sizes': [(128,64,32), (128,64), (64,32), (64,)],
    'model__alpha': [1e-4, 1e-3, 1e-2],
    'model__learning_rate_init': [1e-3, 3e-4],
    'model__activation': ['relu', 'tanh'],
    'model__batch_size': [64, 128, 256],
}

gcv = GridSearchCV(ann, param_grid=param_grid, scoring='neg_mean_absolute_error', cv=3, n_jobs=-1, verbose=1)
gcv.fit(X_train, y_train)

print('Best params:', gcv.best_params_)
print('Best CV MAE:', -gcv.best_score_)

# Show top configs
import pandas as pd, numpy as np
cv = pd.DataFrame(gcv.cv_results_).sort_values('rank_test_score').head(10)
cv[['rank_test_score','mean_test_score','param_model__hidden_layer_sizes','param_model__alpha','param_model__learning_rate_init','param_model__activation','param_model__batch_size']].assign(mean_MAE=lambda d: -d['mean_test_score'])


### Evaluate Best ANN on Test + Plots (Pred vs Actual, Residuals, Residual vs Predicted)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

best_ann = gcv.best_estimator_
y_pred_test = best_ann.predict(X_test)

ann_metrics = {
    'MAE': mean_absolute_error(y_test, y_pred_test),
    'MAPE': mape_masked(y_test, y_pred_test),
    'RMSE': rmse(y_test, y_pred_test),
    'R2': r2_score(y_test, y_pred_test)
}
print(ann_metrics)

def plot_pred_vs_actual(y_true, y_pred, title):
    plt.figure()
    plt.scatter(y_true, y_pred, s=8, alpha=0.5)
    lo = np.percentile(np.r_[y_true, y_pred], 1)
    hi = np.percentile(np.r_[y_true, y_pred], 99)
    plt.plot([lo,hi],[lo,hi])
    plt.xlim([lo,hi]); plt.ylim([lo,hi])
    plt.xlabel('Actual'); plt.ylabel('Predicted'); plt.title(title)
    plt.show()

def plot_residual_hist(y_true, y_pred, title):
    resid = y_pred - y_true
    plt.figure()
    plt.hist(resid, bins=40)
    plt.xlabel('Residual'); plt.ylabel('Count'); plt.title(title + ' — Residuals')
    plt.show()

def plot_residual_vs_pred(y_true, y_pred, title):
    resid = y_pred - y_true
    plt.figure()
    plt.scatter(y_pred, resid, s=8, alpha=0.5)
    plt.axhline(0, linestyle='--')
    plt.xlabel('Predicted'); plt.ylabel('Residual'); plt.title(title + ' — Residual vs Predicted')
    plt.show()

plot_pred_vs_actual(y_test, y_pred_test, 'ANN — Predicted vs Actual (Test)')
plot_residual_hist(y_test, y_pred_test, 'ANN (Test)')
plot_residual_vs_pred(y_test, y_pred_test, 'ANN (Test)')


### Learning Curve (ANN loss)

In [ ]:
# Access the final model's loss curve from the MLPRegressor inside the pipeline
mlp = best_ann.named_steps['model']
if hasattr(mlp, 'loss_curve_'):
    import matplotlib.pyplot as plt
    plt.figure()
    plt.plot(mlp.loss_curve_)
    plt.xlabel('Epoch'); plt.ylabel('Training Loss'); plt.title('ANN Learning Curve')
    plt.show()
else:
    print('No loss_curve_ attribute available.')


### Summary Table — ANN vs KNN/DT/RF (Test)

In [ ]:
import pandas as pd
df_sum = pd.DataFrame(results_baseline).T
df_sum.loc['ANN(best)'] = ann_metrics
df_sum = df_sum[['MAE','MAPE','RMSE','R2']].sort_values('MAE')
df_sum


### Discussion (meets requirement)
- **ANN excels** when non-linear interactions between area/rooms and location (state/city/region) matter.
- **Trees/forests** can be more robust on small/noisy datasets or when category granularity is high.
- Residual plots indicate whether variance grows with price (consider log-target if needed).
- Hyperparameter table justifies the chosen depth/width/activation/learning-rate.


---

## Part 2 — Image Classification (Complete Template)

**Dataset requirement:** 40–50 images per class; ≥10 self-captured per class.  
Expected layout (already created in `data_cls/`):

```
data_cls/
  train/
    classA/ *.jpg
    classB/ *.jpg
  val/
    classA/ *.jpg
    classB/ *.jpg
  test/
    classA/ *.jpg
    classB/ *.jpg
```

Below are two training modes: **from scratch** and **transfer learning**.  
> Install: `pip install torch torchvision`


In [ ]:
# --- Uncomment to run when torch/torchvision are available ---
# import torch, torch.nn as nn, torch.optim as optim
# from torchvision import datasets, transforms, models
# from torch.utils.data import DataLoader
# import numpy as np
# import matplotlib.pyplot as plt
# from pathlib import Path
# from scripts.confusion import plot_confusion_matrix
# from sklearn.metrics import classification_report
#
# DATA_ROOT = Path('data_cls')
# IMG_SIZE = 224; BATCH = 32; EPOCHS = 10; LR = 1e-3
#
# tfm_train = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(5),
#     transforms.ToTensor(),
# ])
# tfm_eval = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.ToTensor(),
# ])
#
# ds_train = datasets.ImageFolder(DATA_ROOT/'train', transform=tfm_train)
# ds_val   = datasets.ImageFolder(DATA_ROOT/'val', transform=tfm_eval)
# ds_test  = datasets.ImageFolder(DATA_ROOT/'test', transform=tfm_eval)
#
# dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True, num_workers=2)
# dl_val   = DataLoader(ds_val, batch_size=BATCH, shuffle=False, num_workers=2)
# dl_test  = DataLoader(ds_test, batch_size=BATCH, shuffle=False, num_workers=2)
#
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#
# # a) FROM SCRATCH (tiny CNN)
# class TinyCNN(nn.Module):
#     def __init__(self, num_classes):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
#             nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
#             nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
#         )
#         self.fc = nn.Linear(64, num_classes)
#     def forward(self, x):
#         x = self.net(x)
#         x = x.view(x.size(0), -1)
#         return self.fc(x)
#
# num_classes = len(ds_train.classes)
# model_scratch = TinyCNN(num_classes).to(device)
# opt = optim.Adam(model_scratch.parameters(), lr=LR)
# crit = nn.CrossEntropyLoss()
#
# def train_epoch(model, dl):
#     model.train(); tot=0; correct=0; loss_sum=0
#     for x,y in dl:
#         x,y = x.to(device), y.to(device)
#         opt.zero_grad(); logits = model(x)
#         loss = crit(logits, y); loss.backward(); opt.step()
#         loss_sum += loss.item()*x.size(0)
#         pred = logits.argmax(1); correct += (pred==y).sum().item(); tot += x.size(0)
#     return loss_sum/tot, correct/tot
#
# def eval_epoch(model, dl):
#     model.eval(); tot=0; correct=0; loss_sum=0
#     preds=[]; trues=[]
#     with torch.no_grad():
#         for x,y in dl:
#             x,y = x.to(device), y.to(device)
#             logits = model(x)
#             loss = crit(logits, y)
#             loss_sum += loss.item()*x.size(0)
#             pred = logits.argmax(1); correct += (pred==y).sum().item(); tot += x.size(0)
#             preds.extend(pred.cpu().numpy()); trues.extend(y.cpu().numpy())
#     return loss_sum/tot, correct/tot, np.array(trues), np.array(preds)
#
# train_losses, val_losses, train_accs, val_accs = [], [], [], []
# for epoch in range(EPOCHS):
#     tr_loss, tr_acc = train_epoch(model_scratch, dl_train)
#     va_loss, va_acc, _, _ = eval_epoch(model_scratch, dl_val)
#     train_losses.append(tr_loss); val_losses.append(va_loss)
#     train_accs.append(tr_acc); val_accs.append(va_acc)
#     print(f"epoch {epoch+1}/{EPOCHS} - train_loss={tr_loss:.4f} val_loss={va_loss:.4f} train_acc={tr_acc:.3f} val_acc={va_acc:.3f}")
#
# # Curves
# plt.figure(); plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val'); plt.legend(); plt.title('Loss'); plt.show()
# plt.figure(); plt.plot(train_accs, label='train'); plt.plot(val_accs, label='val'); plt.legend(); plt.title('Accuracy'); plt.show()
#
# # Confusion matrix on test
# _, _, y_true, y_pred = eval_epoch(model_scratch, dl_test)
# fig = plot_confusion_matrix(y_true, y_pred, ds_train.classes, normalize=True, title='Confusion (Scratch)')
# plt.show()
# print(classification_report(y_true, y_pred, target_names=ds_train.classes))
#
# # b) TRANSFER LEARNING — MobileNetV2
# model_tl = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
# for p in model_tl.features.parameters(): p.requires_grad = False
# in_features = model_tl.classifier[1].in_features
# model_tl.classifier[1] = nn.Linear(in_features, num_classes)
# model_tl = model_tl.to(device)
# opt_tl = optim.Adam(model_tl.parameters(), lr=LR)
#
# # Train TL head
# train_losses, val_losses, train_accs, val_accs = [], [], [], []
# for epoch in range(EPOCHS):
#     tr_loss, tr_acc = train_epoch(model_tl, dl_train)
#     va_loss, va_acc, _, _ = eval_epoch(model_tl, dl_val)
#     train_losses.append(tr_loss); val_losses.append(va_loss)
#     train_accs.append(tr_acc); val_accs.append(va_acc)
#     print(f"TL epoch {epoch+1}/{EPOCHS} - train_loss={tr_loss:.4f} val_loss={va_loss:.4f} train_acc={tr_acc:.3f} val_acc={va_acc:.3f}")
#
# # Evaluate TL on test
# _, _, y_true, y_pred = eval_epoch(model_tl, dl_test)
# fig = plot_confusion_matrix(y_true, y_pred, ds_train.classes, normalize=True, title='Confusion (TL)')
# plt.show()
# print(classification_report(y_true, y_pred, target_names=ds_train.classes))
#
# # Show 5 examples per class with predicted label + confidence
# import torch.nn.functional as F
# shown = {c:0 for c in ds_train.classes}
# model_tl.eval()
# for imgs, labels in dl_test:
#     imgs = imgs.to(device)
#    _log = model_tl(imgs); probs = F.softmax(_log, dim=1).cpu().numpy()
#     preds = probs.argmax(1)
#     for i in range(len(imgs)):
#         cls = ds_train.classes[labels[i].item()]
#         if shown[cls] >= 5: continue
#         conf = probs[i, preds[i]]
#         plt.figure(); plt.imshow(imgs[i].cpu().permute(1,2,0)); plt.axis('off')
#         plt.title(f"Actual: {cls} | Pred: {ds_train.classes[preds[i]]} | conf={conf:.2f}")
#         plt.show()
#         shown[cls]+=1
#     if all(v>=5 for v in shown.values()): break


---

## Part 3 — Object Detection & Segmentation (YOLOv8 Template)

Prepare your data under `data_od/` and edit `data_od/data.yaml` to list your classes.

> Install: `pip install ultralytics`

Two cases required:
1) **Without transfer learning** (train from scratch, e.g., `model = YOLO('yolov8n.yaml')`)  
2) **With transfer learning** (start from COCO pretrained weights, e.g., `yolov8n.pt`)

For segmentation, use a `-seg` model (e.g., `yolov8n-seg.pt`) and provide polygon/seg labels.


In [ ]:
# --- Uncomment to run when ultralytics is installed ---
# from ultralytics import YOLO
# from pathlib import Path
#
# DATA_YAML = Path('data_od/data.yaml')  # edit paths/classes inside
#
# # 1) With transfer learning (detection)
# model = YOLO('yolov8n.pt')
# results = model.train(data=str(DATA_YAML), imgsz=640, epochs=50, project='runs/od', name='det_tl')
# metrics = model.val()
# model.predict(source='data_od/images/test', save=True)
#
# # 2) Without transfer learning (detection) — from scratch
# model_scratch = YOLO('yolov8n.yaml')
# results_s = model_scratch.train(data=str(DATA_YAML), imgsz=640, epochs=50, project='runs/od', name='det_scratch')
# metrics_s = model_scratch.val()
#
# # 3) Segmentation (optional, if you annotated masks)
# model_seg = YOLO('yolov8n-seg.pt')
# results_seg = model_seg.train(data=str(DATA_YAML), imgsz=640, epochs=50, project='runs/od', name='seg_tl')
# metrics_seg = model_seg.val()
# model_seg.predict(source='data_od/images/test', save=True)
